# Lecture 03: High-Performance Data Wrangling & Join Strategies in Spark
This notebook provides a professional, step-by-step developer guide to structured DataFrame operations, partitioned/bucketed storage layout designs, multi-field aggregations, and join optimizations (Sort-Merge vs. Broadcast).

### 1. Initialize SparkSession
We configure a Spark Session optimized for data wrangling, enabling snappy parquet compression and dynamic partition overrides.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Lecture03_DataWrangling") \
    .master("local[2]") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized successfully!")

### 2. Explicit Schema Enforcement
We define strict nested schemas using StructType to optimize performance by avoiding costly file scanning during schema inference.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

tx_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("transactionId", StringType(), False),
    StructField("amount", DoubleType(), True),
    StructField("category", StringType(), True),
    StructField("country", StringType(), True),
    StructField("timestamp_str", StringType(), True)
])
print("Schema enforced definition complete:", tx_schema)

### 3. Generate Mock Transactions Data
We create a list of mock user transaction objects containing anomalies and missing category fields.

In [ ]:
raw_transactions = [
    (101, "tx_9901", 1200.50, "Electronics", "US", "2026-06-05 10:00:00"),
    (102, "tx_9902", 45.99, "Books", "CA", "2026-06-05 10:05:00"),
    (101, "tx_9903", 300.00, None, "US", "2026-06-05 10:10:00"),
    (103, "tx_9904", 1500.00, "Electronics", "DE", "2026-06-05 10:15:00"),
    (104, "tx_9905", 89.50, "Fashion", "US", "2026-06-05 10:20:00"),
    (102, "tx_9906", 12.00, "Books", "CA", "2026-06-05 10:25:00")
]
print(f"Generated {len(raw_transactions)} raw transaction entries.")

### 4. Create DataFrame
Applying our schema to convert our list into a distributed Spark DataFrame and printing its schema structure.

In [ ]:
df = spark.createDataFrame(raw_transactions, schema=tx_schema)
df.printSchema()
df.show(3)

### 5. Writing and Reading CSV with Options
Writing to local filesystem as CSV using headers and custom separators, then reading it back into memory.

In [ ]:
import shutil
csv_path = "data/raw_csv"
if os.path.exists(csv_path): shutil.rmtree(csv_path)

# Write with header and custom delimiter
df.write.mode("overwrite").option("header", "true").option("sep", "|").csv(csv_path)

# Read back with matching options
csv_df = spark.read.option("header", "true").option("sep", "|").schema(tx_schema).csv(csv_path)
csv_df.show(2)

### 6. Partitioned Storage Writes (Parquet)
Writing Parquet files partitioned by the country column. This physically divides the data into subdirectories to optimize query speed for specific regions.

In [ ]:
parquet_path = "data/partitioned_parquet"
if os.path.exists(parquet_path): shutil.rmtree(parquet_path)

# Write partitioned by country
df.write.mode("overwrite").partitionBy("country").parquet(parquet_path)
print("Partitioned subfolders on disk:", os.listdir(parquet_path))

### 7. Reading Partitioned Storage
Spark reads partition paths dynamically and injects the partition key ('country') back as a column in the DataFrame.

In [ ]:
partitioned_df = spark.read.parquet(parquet_path)
partitioned_df.show(3)

### 8. Bucketing and Sorting tables on Disk
Bucketing divides the dataset into a fixed number of buckets based on the hash of a key. This avoids shuffles when joining on the bucketed key.

In [ ]:
# Note: Bucketing requires saving the data as a table in a catalog
spark.sql("DROP TABLE IF EXISTS bucketed_tx")
df.write.mode("overwrite").bucketBy(4, "userId").sortBy("amount").saveAsTable("bucketed_tx")
print("Table created and bucketed successfully!")

### 9. Multi-line JSON processing
Parsing nested JSON documents where each record can span multiple lines.

In [ ]:
json_path = "data/multi_line.json"
if os.path.exists(json_path): shutil.rmtree(json_path)

# Write JSON
df.write.mode("overwrite").json(json_path)

# Read multiline JSON
json_df = spark.read.option("multiLine", "true").json(json_path)
json_df.show(2)

### 10. Column Projections & Type Casting
Selecting columns, casting datatypes explicitly, and aliasing results.

In [ ]:
projected_df = df.select(
    col("userId").alias("user_id"),
    col("amount").cast("integer").alias("rounded_amount"),
    col("category")
)
projected_df.show(3)

### 11. Complex Boolean Logical Filtering
Filtering records by chaining boolean criteria with bitwise operators: `&` (AND), `|` (OR), and `~` (NOT).

In [ ]:
filtered_df = df.filter(
    (col("amount") >= 100.0) &
    ((col("category") == "Electronics") | (col("country") == "US")) &
    col("category").isNotNull()
)
filtered_df.show()

### 12. Column Actions: withColumn & withColumnRenamed
Adding new computed columns (e.g. transaction fee calculation) and renaming key identifiers.

In [ ]:
updated_df = df.withColumn("tx_fee", col("amount") * 0.02) \
               .withColumnRenamed("transactionId", "tx_id")
updated_df.show(3)

### 13. Basic GroupBy Aggregation
Grouping transactions by country and calculating simple total sum values.

In [ ]:
df.groupBy("country").sum("amount").show()

### 14. Advanced Multi-Aggregations
Grouping by country and computing total revenue, average amount, max transaction value, and distinct active users counts using HyperLogLog algorithm.

In [ ]:
from pyspark.sql.functions import avg, max, approx_count_distinct

df.groupBy("country").agg(
    sum("amount").alias("total_revenue"),
    avg("amount").alias("average_transaction"),
    max("amount").alias("highest_sale"),
    approx_count_distinct("userId").alias("approx_unique_customers")
).show()

### 15. Inner Join Execution
Merging transaction records with lookup users table matching on primary key identifier.

In [ ]:
users_df = spark.createDataFrame([
    (101, "Alice", "Premium"),
    (102, "Bob", "Standard"),
    (103, "Charlie", "Premium")
], ["userId", "userName", "tier"])

df.join(users_df, "userId", "inner").show()

### 16. Left Outer Join
Retaining transactions even if the customer lookup record is missing.

In [ ]:
df.join(users_df, "userId", "left").show()

### 17. Cross Join Cartesian execution
Performing cartesian products. Useful for generating permutations, but should be used carefully on large datasets to avoid memory bottlenecks.

In [ ]:
small_lookup = spark.createDataFrame([("FlagA",), ("FlagB",)], ["flag"])
df.crossJoin(small_lookup).show(4)

### 18. Default Join optimization plan (SortMergeJoin)
Joining two datasets. Look for the SortMergeJoin operator and shuffles (Exchange) in the plan.

In [ ]:
df.join(users_df, "userId").explain()

### 19. Broadcast Join optimization (BroadcastHashJoin)
Broadcasting the smaller lookup table to all executors, skipping network shuffles completely.

In [ ]:
from pyspark.sql.functions import broadcast
df.join(broadcast(users_df), "userId").explain()

### 20. Self-Join Analysis
Joining a dataset with itself. Useful for analyzing sequence paths, parent-child relations, or offset timestamps.

In [ ]:
df_self_1 = df.alias("t1")
df_self_2 = df.alias("t2")

df_self_1.join(
    df_self_2,
    (col("t1.userId") == col("t2.userId")) &
    (col("t1.transactionId") != col("t2.transactionId")),
    "inner"
).select("t1.userId", "t1.transactionId", "t2.transactionId").show(4)

### 21. Stop Spark Session
Cleaning session allocations.

In [ ]:
spark.stop()